# CROCUS WXT Quicklook — ESS-DIVE Archive

Quicklook plots for a NEIU rooftop WXT weather-station archive built from **ESS-DIVE** CROCUS data (see the `ess-dive` download notebook for how `NEIU_wxt_essdive.nc` is produced).

These files are **native 10-second** data and carry **no unit metadata**, so units are supplied below. This is the ESS-DIVE counterpart to the Sage WXT quicklook — same kinds of panels (time series + wind rose), adapted to this source's variable names (`wind_dir_10s`, `wind_mean_10s`, `wind_max_10s`) and cadence.

Two data notes:
- **Rainfall is a cumulative gauge total**, not a per-interval amount, so it's plotted as an accumulating curve. An optional rain-*rate* panel follows.
- The record has a few real gaps (instrument outages). We insert `NaN` across any gap longer than 60 s so plot lines break cleanly instead of drawing straight across missing time.

## Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

ARCHIVE = "essdive"        # folder holding the *_essdive.nc archives
WXT_FILE = "NEIU_wxt_essdive.nc"

# ESS-DIVE WXT files carry no units; supply them here (fix in one place if needed)
UNITS = {
    "temperature": "°C", "dewpoint": "°C", "wetbulb": "°C",
    "humidity": "%", "pressure": "hPa", "rainfall": "mm (cumulative)",
    "wind_mean_10s": "m s⁻¹", "wind_max_10s": "m s⁻¹", "wind_dir_10s": "deg",
}
GAP_S = 60          # break plot lines across gaps longer than this (seconds)

wxt = xr.open_dataset(os.path.join(ARCHIVE, WXT_FILE))

def break_gaps(da, time, gap_s=GAP_S):
    """Insert NaN where the time step exceeds gap_s, so lines don't
    draw straight across data gaps. Returns a float array copy."""
    y = da.values.astype(float).copy()
    dt = np.diff(time).astype("timedelta64[s]").astype(float)
    y[1:][dt > gap_s] = np.nan
    return y

t = wxt.time.values
site = wxt.attrs.get("site_ID", wxt.attrs.get("site_abbr", "NEIU"))
print(f"{site} WXT  {str(t[0])[:16]} -> {str(t[-1])[:16]}  ({len(t):,} records)")

## Time-series stack

Temperature/dewpoint/wetbulb together, then humidity, pressure, cumulative rainfall, and wind speed (mean with max overlaid).

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True)

for v, c in [("temperature", "tab:red"), ("dewpoint", "tab:green"),
             ("wetbulb", "tab:blue")]:
    axes[0].plot(t, break_gaps(wxt[v], t), lw=0.5, color=c, label=v)
axes[0].set_ylabel(f"temp [{UNITS['temperature']}]")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].plot(t, break_gaps(wxt.humidity, t), lw=0.5, color="tab:cyan")
axes[1].set_ylabel(f"RH [{UNITS['humidity']}]")

axes[2].plot(t, break_gaps(wxt.pressure, t), lw=0.5, color="tab:purple")
axes[2].set_ylabel(f"P [{UNITS['pressure']}]")

axes[3].plot(t, break_gaps(wxt.rainfall, t), lw=0.6, color="tab:blue")
axes[3].set_ylabel(f"rain\n[{UNITS['rainfall']}]")

axes[4].plot(t, break_gaps(wxt.wind_mean_10s, t), lw=0.5, color="0.4", label="mean")
axes[4].plot(t, break_gaps(wxt.wind_max_10s, t), lw=0.3, color="tab:orange",
             alpha=0.6, label="max")
axes[4].set_ylabel(f"wind [{UNITS['wind_mean_10s']}]")
axes[4].legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("time (UTC)")
fig.suptitle(f"CROCUS WXT (ESS-DIVE) — {site}", y=0.995)
fig.tight_layout()
plt.show()

## Optional: rainfall rate

The cumulative gauge total differenced into per-interval rain. Negatives are clipped to zero, which removes the occasional drop where the gauge resets or the record jumps across a gap.

In [ ]:
rain_rate = np.diff(wxt.rainfall.values, prepend=np.nan)
rain_rate = np.clip(rain_rate, 0, None)        # drop gauge resets / gap jumps
rain_rate[1:][np.diff(t).astype("timedelta64[s]").astype(float) > GAP_S] = np.nan

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, rain_rate, lw=0.5, color="tab:blue")
ax.set_ylabel("rain rate\n[mm / 10 s]"); ax.set_xlabel("time (UTC)")
ax.set_title(f"WXT rainfall rate — {site}")
fig.tight_layout(); plt.show()

## Wind rose

Built with plain matplotlib polar bars (no `windrose` package needed). Calm periods (speed = 0) are dropped, since wind direction is undefined then. Petals point in the compass direction the wind comes *from*, stacked and colored by speed bin.

In [ ]:
def wind_rose(direction, speed, nsect=16, sbins=(0,1,2,3,5,8,12), cmap="viridis"):
    """Frequency wind rose: petals by direction, stacked by speed bin (% of obs)."""
    d = np.asarray(direction, float); s = np.asarray(speed, float)
    m = np.isfinite(d) & np.isfinite(s) & (s > 0)
    d, s = d[m], s[m]
    width = 2*np.pi / nsect
    edges = np.linspace(0, 360, nsect+1)
    centers = np.deg2rad(edges[:-1] + 360/nsect/2)
    sbins = np.array(sbins)
    colors = plt.get_cmap(cmap)(np.linspace(0.15, 0.95, len(sbins)))

    ax = plt.subplot(111, projection="polar")
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
    bottom = np.zeros(nsect)
    for k in range(len(sbins)):
        lo = sbins[k]; hi = sbins[k+1] if k+1 < len(sbins) else np.inf
        counts = np.array([
            ((d >= edges[i]) & (d < edges[i+1]) & (s >= lo) & (s < hi)).sum()
            for i in range(nsect)]) / len(s) * 100
        ax.bar(centers, counts, width=width*0.9, bottom=bottom, color=colors[k],
               edgecolor="white", linewidth=0.3,
               label=f"{lo:g}-{hi:g}" if np.isfinite(hi) else f">={lo:g}")
        bottom += counts
    ax.set_title(f"Wind rose — {site} (% of obs)")
    ax.legend(title="m s$^{-1}$", bbox_to_anchor=(1.15, 1.05), fontsize=8)
    return ax

plt.figure(figsize=(7, 7))
wind_rose(wxt.wind_dir_10s.values, wxt.wind_mean_10s.values)
plt.tight_layout(); plt.show()